In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc numpy
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     token="<your-api-key>",
#     instance="<your-crn>",
#     overwrite=True
# )

# Sledování nebo zrušení úlohy

Zobraz seznam svých pracovních zátěží na [stránce Workloads](https://quantum.cloud.ibm.com/workloads).

## Zobrazení stavu úlohy
Přejdi do své [tabulky Workloads](https://quantum.cloud.ibm.com/workloads) a zkontroluj sloupec Status, zda byla úloha dokončena nebo selhala.

## Zobrazení zbývajícího využití
Přejdi do své [tabulky Instances](https://quantum.cloud.ibm.com/instances) a vyber záložku odpovídající plánu, pro který chceš zobrazit zbývající využití. Zobrazí se celkový použitý čas a celkový zbývající čas na tvém plánu.

## Zobrazení metrik počtu odeslaných úloh a pracovních zátěží
Přejdi na [stránku Analytics](https://quantum.cloud.ibm.com/analytics), kde uvidíš celkový počet odeslaných úloh a také počet dávkových pracovních zátěží a Session pracovních zátěží. Upozorňujeme, že stránku Analytics vidíš pouze pro účty, které vlastníš nebo spravuješ.

## Sledování úlohy
Použij instanci úlohy ke kontrole stavu úlohy nebo k načtení výsledků zavoláním příslušného příkazu:

|                               |                                                                                                                                                                                                              |
| ----------------------------- | ------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------ |
| job.result()                  | Zobraz výsledky úlohy ihned po jejím dokončení. Výsledky úlohy jsou dostupné po jejím dokončení. Proto je `job.result()` blokující volání až do dokončení úlohy.                               |
| job.job\_id()                 | Vrátí ID, které jednoznačně identifikuje danou úlohu. Načtení výsledků úlohy později vyžaduje ID úlohy. Proto se doporučuje ukládat ID úloh, které si možná budeš chtít načíst později. |
| job.status()                  | Zkontroluj stav úlohy.                                                                                                                                                                                        |
| job = service.job(\<job\_id>) | Načti úlohu, kterou jsi dříve odeslal(a). Toto volání vyžaduje ID úlohy.                                                                                                                                      |

<span id="retrieve-later"></span>
## Načtení výsledků úlohy později
Zavolej `service.job(\<job\_id>)` pro načtení úlohy, kterou jsi dříve odeslal(a). Pokud nemáš ID úlohy nebo chceš načíst více úloh najednou — včetně úloh z vyřazených QPU (quantum processing units) — zavolej místo toho `service.jobs()` s volitelnými filtry. Viz [QiskitRuntimeService.jobs](../api/qiskit-ibm-runtime/qiskit-runtime-service#jobs).

> **Note:** `service.jobs()` také vrací úlohy spuštěné ze zastaralého balíčku `qiskit-ibm-provider`. Úlohy odeslané starším (také zastaralým) balíčkem `qiskit-ibmq-provider` již nejsou dostupné.

### Příklad
Tento příklad vrátí 10 nejnovějších runtime úloh, které byly spuštěny na `my_backend`:

In [1]:
# This cell is hidden from users
from qiskit import QuantumCircuit
from qiskit.circuit import Parameter
from qiskit.transpiler import generate_preset_pass_manager

from qiskit_ibm_runtime import QiskitRuntimeService, SamplerV2
import numpy as np


my_backend = "ibm_torino"
service = QiskitRuntimeService()
# backend = service.backend(my_backend)
backend = service.least_busy()

# Define two circuits, each with one parameter with two parameters.
circuit = QuantumCircuit(2)
circuit.h(0)
circuit.cx(0, 1)
circuit.ry(Parameter("a"), 0)
circuit.cx(0, 1)
circuit.h(0)
circuit.measure_all()


pm = generate_preset_pass_manager(optimization_level=1, backend=backend)
transpiled_circuit = pm.run(circuit)

params = np.random.uniform(size=(2, 3)).T

sampler_pub = (transpiled_circuit, params)

# Instantiate the new estimator object, then run the transpiled circuit
# using the set of parameters and observables.
sampler = SamplerV2(mode=backend)
job = sampler.run([sampler_pub], shots=4)
print(job.job_id())

d305ck0ocacs73ajagvg


In [2]:
result = job.result()


spans = job.result().metadata["execution"]["execution_spans"]
print(spans)

ExecutionSpans([DoubleSliceSpan(<start='2025-09-09 16:31:16', stop='2025-09-09 16:31:16', size=24>)])


In [3]:
params = np.random.uniform(size=(2, 3))
params

array([[0.2260416 , 0.8747859 , 0.44361995],
       [0.94700856, 0.96826017, 0.98426562]])

In [4]:
mask = spans[0].mask(0)
mask

array([[[ True,  True,  True,  True],
        [ True,  True,  True,  True]],

       [[ True,  True,  True,  True],
        [ True,  True,  True,  True]],

       [[ True,  True,  True,  True],
        [ True,  True,  True,  True]]])

In [ ]:
from qiskit_ibm_runtime import QiskitRuntimeService

# Initialize the account first.
service = QiskitRuntimeService()
# Use `limit` to retrieve a specific number of jobs. The default `limit` is 10.
service.jobs(backend_name=my_backend)

## Cancel a job

You can cancel a job from the IBM Quantum Platform dashboard either on the Workloads page or the details page for a specific workload. On the Workloads page, click the overflow menu at the end of the row for that workload, and select Cancel. If you are on the details page for a specific workload, use the Actions dropdown at the top of the page, and select Cancel.

In Qiskit, use `job.cancel()` to cancel a job.

<span id="execution-spans"></span>
## View Sampler execution spans

The results of [`SamplerV2`](/docs/api/qiskit-ibm-runtime/sampler-v2) jobs executed in Qiskit Runtime contain execution timing information in their metadata.
This timing information can be used to place upper and lower timestamp bounds on when particular shots were executed on the QPU.
Shots are grouped into [`ExecutionSpan`](/docs/api/qiskit-ibm-runtime/execution-span-execution-span) objects, each of which indicates a start time, a stop time, and a specification of which shots were collected in the span.

An execution span specifies which data was executed during its window by providing an [`ExecutionSpan.mask`](/docs/api/qiskit-ibm-runtime/execution-span-execution-span#mask) method. This method, given any [Primitive Unified Block (PUB)](/docs/guides/primitive-input-output) index, returns a boolean mask that is `True` for all shots executed during its window. PUBs are indexed by the order in which they were given to the Sampler run call. If, for example, a PUB has shape `(2, 3)` and was run with four shots, then the mask's shape is `(2, 3, 4)`. See the [execution_span](/docs/api/qiskit-ibm-runtime/execution-span) API page for full details.

Example:

To view execution span information, review the metadata of the result returned by `SamplerV2`, which comes in the form of an `ExecutionSpans` object. This object is a list-like container containing instances of subclasses of `ExecutionSpan`, such as `SliceSpan`:

In [6]:
from qiskit.primitives import BitArray

# Get the mask of the 1st PUB for the 0th span.
mask = spans[0].mask(0)

# Decide whether the 0th shot of parameter set (1, 2) occurred in this span.
in_this_span = mask[2, 1, 0]

# Create a new bit array containing only the PUB-1 data collected during this span.
bits = result[0].data.meas
filtered_data = BitArray(bits.array[mask], bits.num_bits)

Execution spans can be filtered to include information pertaining to specific PUBs, selected by their indices:

In [7]:
# take the subset of spans that reference data in PUBs 0 or 2
spans.filter_by_pub([0, 2])

ExecutionSpans([DoubleSliceSpan(<start='2025-09-09 16:31:16', stop='2025-09-09 16:31:16', size=24>)])

View global information about the collection of execution spans:

In [8]:
print("Number of execution spans:", len(spans))
print("  Start of the first span:", spans.start)
print("     End of the last span:", spans.stop)
print("       Total duration (s):", spans.duration)

Number of execution spans: 1
  Start of the first span: 2025-09-09 16:31:16.320568
     End of the last span: 2025-09-09 16:31:16.865858
       Total duration (s): 0.54529


Extract and inspect a particular span:

In [9]:
spans.sort()
print(" Start of first span:", spans[0].start)
print("   End of first span:", spans[0].stop)
print("#shots in first span:", spans[0].size)

 Start of first span: 2025-09-09 16:31:16.320568
   End of first span: 2025-09-09 16:31:16.865858
#shots in first span: 24


## Zrušení úlohy
Úlohu můžeš zrušit z řídicího panelu IBM Quantum Platform buď na stránce Workloads, nebo na stránce s podrobnostmi konkrétní pracovní zátěže. Na stránce Workloads klikni na nabídku přetečení na konci řádku dané pracovní zátěže a vyber Zrušit. Pokud jsi na stránce s podrobnostmi konkrétní pracovní zátěže, použij rozbalovací nabídku Actions v horní části stránky a vyber Zrušit.

V Qiskit použij `job.cancel()` pro zrušení úlohy.

<span id="execution-spans"></span>
## Zobrazení Sampler execution spans
Výsledky úloh [`SamplerV2`](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/sampler-v2) spuštěných v Qiskit Runtime obsahují ve svých metadatech informace o časování provádění.
Tyto informace o časování lze použít k určení horní a dolní časové hranice, kdy byly konkrétní snímky provedeny na QPU.
Snímky jsou seskupeny do objektů [`ExecutionSpan`](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/execution-span-execution-span), z nichž každý udává čas začátku, čas konce a specifikaci toho, které snímky byly v daném rozsahu sesbírány.

Execution span určuje, která data byla provedena během jejího okna, prostřednictvím metody [`ExecutionSpan.mask`](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/execution-span-execution-span#mask). Tato metoda, pro libovolný index [Primitive Unified Block (PUB)](/guides/primitive-input-output), vrací booleovskou masku, která je `True` pro všechny snímky provedené během tohoto okna. PUBs jsou indexovány podle pořadí, v jakém byly předány volání Sampler run. Pokud má například PUB tvar `(2, 3)` a byl spuštěn se čtyřmi snímky, pak tvar masky je `(2, 3, 4)`. Úplné podrobnosti najdeš na stránce API [execution_span](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/execution-span).

Příklad:
Pro zobrazení informací o execution span si prohlédni metadata výsledku vráceného `SamplerV2`, která mají podobu objektu `ExecutionSpans`. Tento objekt je kontejner podobný seznamu, který obsahuje instance podtříd `ExecutionSpan`, například `SliceSpan`: